<a href="https://colab.research.google.com/github/MakWigglz/MaksPortfolio/blob/Mak's-main/financial_calculations_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
def calculate_returns(df, periods=None):
    if periods is None:
        periods = ['1d', '1w', '1m', '3m', '6m', '1y', 'ytd']
    returns = {}
    prices = df['Close'].values
    if len(prices) > 0:
        current_price = prices[-1]
        if len(prices) > 1 and '1d' in periods:
            returns['1d'] = (current_price / prices[-2] - 1) * 100
        if len(prices) > 5 and '1w' in periods:
            returns['1w'] = (current_price / prices[-6] - 1) * 100
        if len(prices) > 21 and '1m' in periods:
            returns['1m'] = (current_price / prices[-21] - 1) * 100
        if len(prices) > 63 and '3m' in periods:
            returns['3m'] = (current_price / prices[-63] - 1) * 100
        if len(prices) > 126 and '6m' in periods:
            returns['6m'] = (current_price / prices[-127] - 1) * 100
        if len(prices) > 252 and '1y' in periods:
            returns['1y'] = (current_price / prices[-253] - 1) * 100
        if 'ytd' in periods and df['Date'].dt.year.nunique() > 1:
            start_of_year_idx = df[df['Date'].dt.year == df['Date'].dt.year.max()].index[0]
            start_of_year_price = df.loc[start_of_year_idx, 'Close']
            returns['ytd'] = (current_price / start_of_year_price - 1) * 100
        return returns

In [ ]:
def calculate_risk_metrics(df):
    """
    Calculate basic risk metrics

    Args:
        df (pandas.DataFrame): Stock price data with 'Close' column

    Returns:
        dict: Dictionary of risk metrics
    """
    metrics = {}

    # Calculate daily returns
    returns = df['Close'].pct_change().dropna()

    # Annualized volatility
    metrics['volatility'] = returns.std() * np.sqrt(252) * 100  # Annualized and in percentage

    # Sharpe ratio (assuming risk-free rate of 0 for simplicity)
    mean_return = returns.mean()
    metrics['sharpe_ratio'] = (mean_return * 252) / (returns.std() * np.sqrt(252)) if returns.std() != 0 else 0

    # Maximum drawdown
    cum_returns = (1 + returns).cumprod()
    running_max = cum_returns.cummax()
    drawdown = (cum_returns / running_max - 1)
    metrics['max_drawdown'] = drawdown.min() * 100  # In percentage

    # Downside deviation (returns below 0)
    negative_returns = returns[returns < 0]
    metrics['downside_deviation'] = negative_returns.std() * np.sqrt(252) * 100 if len(negative_returns) > 0 else 0

    # Sortino ratio (using 0 as minimum acceptable return)
    metrics['sortino_ratio'] = (mean_return * 252) / (metrics['downside_deviation'] / 100) if metrics['downside_deviation'] != 0 else 0

    return metrics